## Lernziel

Schema-Bindings in einem isolierten Ausgabeordner erzeugen, Unterschiede prüfen und die spätere Aktivierung vorbereiten.

**Voraussetzungen:** Gespeicherte PEExcel-Exporte; keine Excel-Installation erforderlich.

Einmalig im Repository installieren: `python -m pip install -e ".[publication]"`.
Zum eigenen Ausführen das Notebook im geklonten Repository öffnen und die Zellen von oben nach unten ausführen.
Die Website zeigt gespeicherte Ausgaben; sie führt keinen Python-Code im Browser aus.

[Tutorial-Notebook herunterladen](downloads/tutorial_schema.ipynb) · [Alle Tutorials](python.qmd)


## 1. Version und Quelle festlegen

Ein neues Excel-Schema verändert Variablendefinitionen, Zeitreihen und Berichte.
Wir verwenden einen vorhandenen Schemaexport und schreiben ausschließlich in einen lokalen Prüfbereich.
Das aktive Paket-Schema wird dabei nicht überschrieben.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Im geklonten Repository ausführen, z. B. aus website/ oder notebooks/.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pexl").is_dir() and (p / "pyproject.toml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Bitte das Notebook innerhalb des peexcel-Repositorys öffnen.")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
OUTPUT = ROOT / "notebooks/local_exports/tutorials"
OUTPUT.mkdir(parents=True, exist_ok=True)
from pexl import schema
from pexl.schema.current import SCHEMA_VERSION
VERSION = "1.13.1_dev"
QUELLE = ROOT / f"data/exports/schema_{VERSION}.xlsx"
ALT = ROOT / "data/schemas/v1_12_7"
PRUEFUNG = OUTPUT / "schema_pruefung"
assert QUELLE.is_file() and ALT.is_dir()
print("Aktives Schema:", SCHEMA_VERSION)
print("Zu prüfender Export:", QUELLE.name)

Aktives Schema: 1_13_1_dev
Zu prüfender Export: schema_1.13.1_dev.xlsx


## 2. Bindings getrennt erzeugen

`dataset_dir` und `generated_dir` sind beide ausdrücklich gesetzt. Ohne `generated_dir`
würde die Funktion in das Paketverzeichnis schreiben. Wiederholung ersetzt nur Dateien im lokalen Prüfbereich.

In [2]:
ergebnis = schema.create(
    QUELLE, dataset_dir=PRUEFUNG / "daten", generated_dir=PRUEFUNG / "generated",
    version=VERSION, replace_files=True,
)
display(pd.Series({"Schema": ergebnis.schema_module.name,
                   "Zeitreihen": ergebnis.timeseries_module.name,
                   "Reports": ergebnis.report_module.name}))

Schema             excel_1_13_1_dev.py
Zeitreihen    timeseries_1_13_1_dev.py
Reports          reports_1_13_1_dev.py
dtype: object

## 3. Unterschiede prüfen

Die vorhandene Vergleichsfunktion prüft die Dataset-Verzeichnisse. Hinzugefügte oder entfernte Variablen,
Einheiten und Metadaten müssen fachlich geprüft werden; erfolgreiche Codegenerierung allein reicht nicht.

In [3]:
from pexl.validate.schema import diff_schema_dirs, schema_diff_to_markdown
from IPython.display import Markdown
unterschiede = diff_schema_dirs(old_dir=ALT, new_dir=ergebnis.dataset_dir)
vergleich = schema_diff_to_markdown(unterschiede, value_format="None")
(PRUEFUNG / "vergleich.md").write_text(vergleich, encoding="utf-8")
# In der Seite nur einen Auszug anzeigen; der vollständige Bericht liegt lokal vor.
display(Markdown(vergleich[:5000]))

# Schema Diff

- **Old:** `C:\Users\schneids\code\peexcel\data\schemas\v1_12_7`
- **New:** `C:\Users\schneids\code\peexcel\notebooks\local_exports\tutorials\schema_pruefung\daten`
- **Key:** `None`

## Dataset files

- Old file count: 3
- New file count: 5
- Added: ['CHART_metadata', 'CHART_types']
- Removed: []

### Warnings
- ⚠ Tables added to schema: ['CHART_metadata', 'CHART_types']

## IN

### Columns
- Added: []
- Removed: []
- Common count: 13

### Rows
Old count: 691
New count: 692
- **New Error count: 6**
- Added/~~Removed~~ var_name:
   - PV_roof_area_percentage



### Potential Errors (#)

| var_name | column | error |
|---|---|---|
| fGHG_grid_column | Formel | VERGLEICH(fGHG_grid_profile;fGHG[#Kopfzeilen];0) |
| fPE_grid_column | Formel | VERGLEICH(fPE_grid_profile;fPE[#Kopfzeilen];0) |
| flex_Signals_selected_column | Formel | VERGLEICH(FLEX_signal_name;Signals[#Kopfzeilen];0) |
| rcp1_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp1_renewable;#NV);rcp1_fossil) |
| rcp2_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp2_renewable;#NV);rcp2_fossil) |
| rcp3_dh | Formel | WENNFEHLER(WENN(FINDEN("rneuerbar";SVERWEIS(heat_th2_carrier_type;cf_constant;7;FALSCH));rcp3_renewable;#NV);rcp3_fossil) |





## OUT

### Columns
- Added: []
- Removed: []
- Common count: 13

### Rows
Old count: 452
New count: 512
- **New Error count: 4**
- Added/~~Removed~~ var_name:
   - EUIdhw_1el
  - EUIdhw_2el
  - EUIdhw_th_total
  - Ec_1el
  - Ec_3el
  - Eh_1el
  - Eh_3el
  - PEI_cf_density_neg
  - PEI_cf_density_pos
  - PEI_importexport_balance_test
  - PEI_virtual_balance_test
  - PEI_virtual_demand
  - PEI_virtual_supply
  - PV_peak_grid_feedin
  - PV_peak_grid_feedin_date
  - Qc_distr_losses
  - Qc_min_0fc
  - Qced_1el
  - Qced_2th
  - Qced_3el
  - Qdhw_1_distr_losses
  - Qdhw_1_drawoff
  - Qdhw_1_generation_losses
  - Qdhw_1_storage_losses
  - Qdhw_1_tap
  - Qdhw_1_total
  - Qdhw_2_distr_losses
  - Qdhw_2_drawoff
  - Qdhw_2_generation_losses
  - Qdhw_2_storage_losses
  - Qdhw_2_tap
  - Qdhw_2_total
  - Qdhw_distr_losses
  - Qdhw_drawoff
  - Qdhw_generation_losses
  - Qdhw_storage_charge
  - Qdhw_storage_losses
  - Qdhw_tap
  - Qenv_c
  - Qenv_c_1el
  - Qenv_c_3el
  - Qenv_dhw
  - Qenv_dhw_1
  - Qenv_dhw_2
  - Qenv_h
  - Qenv_h_1el
  - Qenv_h_3el
  - Qh_distr_losses
  - Qh_flex_wasteheat
  - Qh_min_wasteheat
  - Qh_wasteheat
  - Qheb_1el
  - Qheb_2th
  - Qheb_3el
  - Qheb_4th
  - Qloss_c_generation
  - Qloss_c_th2_generation
  - Qloss_h_generation
  - Qloss_h_th2_generation
  - Qloss_h_th4_generation
  - test_EUI_balance
  - test_GWP_sum
  - test_heat_balance_seasonal_mismatch
  - test_legacy_GWP_sum
  - ~~Edhw_1_th~~
  - ~~Edhw_2_th~~
  - ~~Edhw_th~~
  - ~~PEI_cf_density~~


### Potential Errors (#)

| var_name | column | error |
|---|---|---|
| GWP_emInt_PV_feedin | Formel | WENNFEHLER(GWP_oe_grid_feedin * NFA_to_GFA_ratio/PV_to_Egrid * 1000;#NV) |
| GWP_emInt_batt_charge | Formel | WENNFEHLER(GWP_oe_battery_charge/Batt_total_charge_input * 1000 * NFA_to_GFA_ratio;#NV) |
| GWP_emInt_ev_charge | Formel | WENNFEHLER(GWP_oe_mob_ev/EUIev_el_total * 1000 * NFA_to_GFA_ratio;#NV) |
| GWP_emInt_flex | Formel | WENNFEHLER(GWP_oe_flex_build * NFA_to_GFA_ratio/(VRGrid_to_flex-VRGrid_to_Eev_flex+VRGrid_to_min-VRGrid_to_Eev_min) * 1000;#NV) |





## SIM

### Columns
- Added: []
- Removed: []
- Common count: 5

### Rows
Old count: 394
New count: 401
- **New Error count: 3**
- Added/~~Removed~~ var_name:
   - DHW_1_tap_kW
  - DHW_2_tap_kW
  - Ec_1el
  - Ec_3el
  - Edhw_1_aux_el
  - Edhw_1el
  - Edhw_2_aux_el
  - Edhw_2el
  - Eh_1el
  - Eh_3el
  - Qenv_c_1el
  - Qenv_c_3el
  - Qenv_dhw_1
  - Qenv_dhw_2
  - Qenv_h_1el
  - Qenv_h_3el
  - Qh_distr_losses
  - Qhed_1el
  - Qhed_2th
  - Qhed_3el
  - Qhed_4th
  - Qhed_total
  - Spalte14
  - Test
  - ~~Batt_max_energy_output~~
  - ~~QH_distr_losses~~
  - ~~Qh_u2~~
  - ~~Qheb_1el~~
  - ~~Qheb_2th~~
  - ~~Qheb_3el~~
  - ~~Qheb_4th~~
  - ~~Qheb_total~~
  - ~~Spalte192~~
  - ~~Spalte20~~
  - ~~Spalte21~~
  - ~~Spalte80~~
  - ~~Spalte81~~
  - ~~Spalte82~~
  - ~~Spalte83~~
  - ~~Spalte84~~
  - ~~Spalte85~~


### Potential Errors (#)

| var_name | column | error |
|---|---|---|
| Signal | Formula | =INDEX(Signals,ROW()-ROW(Signals[#Headers]),flex_Signals_selected_column) |
| Ti0cooled | Formula | =IF(NFA_cooled/NFA_total>0,Tsetheat_min,#N/A) |
| Ti0uncooled | Formula | =IF(1-NFA_cooled/NFA_total>0,Tsetheat_min,#N/A) |





## CHART_metadata

- Status: **added**
- Rows: {'count': 73}
- Columns: ['chart_name', 'label_de', 'var_name', 'role', 'order', 'color', 'pattern']

## CHART_types

- Status: **added**
- Rows: {'count': 7}
- Columns: ['chart_name', 'tab_name', 'title', 'chart_type']

## Audit summary

- Raw audit present (old): True
- Raw audit present (new): True

### Empty var_name exclusions (converter)

**Old:**
```json
{'IN': {'empty_var_name_count': 94}, 'OUT': {'empty_v

## 4. Aktivierung bewusst vornehmen

`schema.create` validiert bereits den Import der erzeugten Module. Nach fachlicher Prüfung werden
Dataset und generierte Module in die vorgesehenen Versionsverzeichnisse übernommen und die Importe
in `src/pexl/schema/current.py` angepasst. Dies ist eine gesonderte Repository-Änderung, kein Nebeneffekt dieses Tutorials.

Anschließend Kernel neu starten, repräsentative Projektexporte einlesen und im Repository
`python -m pytest` ausführen. Dabei Importwarnungen, neue Variablen und Standardberichte kontrollieren.
Bei reiner Reportpflege die bestehende Codegenerierung verwenden, keine zweite Berichtsregistry aufbauen.

## Ergebnis

Du erhältst geprüfte Python-Module und einen Vergleichsbericht im lokalen Prüfbereich.
Das aktive Schema bleibt unverändert, bis du die geprüfte Version ausdrücklich übernimmst.



## Weiterarbeiten

[Zur Tutorial-Übersicht](python.qmd). Die ursprünglichen Arbeitsnotebooks bleiben erhalten: [howto_schema_transition](https://github.com/simonschaluppe/peexcel/blob/master/notebooks/howto_schema_transition.ipynb).
